# Module 3 · Session 3 — Lab Notebook
## Output Control: Structured Outputs & Prompt Influence — "The Reliability Bench"

**Course:** Generative & Agentic AI Systems
**Time:** ~45 minutes (in-session) · seeds this session's assignment
**Runs on:** Google Colab, Kaggle, or your own laptop — no GPU, no API key needed

---

### New to this track? Read this first 👋

This notebook assumes you've done the previous labs, but here's a 30-second refresher on the jargon we'll use a lot today:

- **Prompt** — the text you send to the model
- **JSON** — a text format for structured data, like `{"name": "Alice", "age": 30}`. Almost every app that talks to an LLM wants the answer back in this format so code can read it.
- **Schema** — a strict definition of what a valid JSON object must look like (which keys, what types)
- **Compliance rate** — out of N attempts, what fraction actually produced valid, on-schema JSON? This is the number we'll measure all lab long.
- **Temperature / top-p** — sampling knobs from Module 2 that control how "random" the model's word choices are

If any term below is unfamiliar, it's probably defined the first time it appears — search this notebook for it.

### What you'll do
| Part | What | Time |
|---|---|---|
| 1 | Build the pipeline — prompt → generate → parse → validate, with retries | 10 min |
| 2 | A/B test 4 prompt styles and measure which one actually works | 15 min |
| 3 | Tune temperature/top-p and chart compliance vs setting | 15 min |
| 4 | Write a short reliability report combining your best findings | 5 min |

Every part builds on the last, so run cells **top to bottom, in order**, even during the exercises.

In [ ]:
# STEP 1: Install the libraries we need
# %pip installs work the same in Colab, Kaggle, and locally — this cell is safe to re-run.
%pip install -q torch transformers matplotlib numpy pandas

# STEP 2: Import everything and load the model
# GPT-2 is a small, free, local model — same one used in previous sessions.
# It has NO built-in "JSON mode" (that's a feature of newer commercial APIs) —
# which is exactly why this lab is useful: you'll build the guardrails BY HAND,
# so you understand what JSON mode is automating for you later (Module 4).

import json
import re
import time
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer

torch.manual_seed(42)

print("Loading GPT-2... (first run downloads ~500MB, later runs are instant)")
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("Done! Model loaded and ready.")

---
## Part 1 — Build the Pipeline (10 min)

A **structured output pipeline** is just four steps in a row:

```
prompt  →  model generates text  →  try to parse it as JSON  →  check it matches your schema
```

If any step fails, we retry (up to a limit) before giving up. Let's build this one function at a time so each piece is easy to understand.

In [ ]:
# A small helper you'll reuse all lab: generate text from a prompt.
# This is the same generate() call from Module 2 — nothing new here.
def generate(prompt, max_new_tokens=60, temperature=0.7, top_p=0.9, do_sample=True):
    ids = tok.encode(prompt, return_tensors="pt")
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tok.eos_token_id,
    )
    # Only return the NEW text (cut off the prompt itself)
    return tok.decode(out[0][ids.shape[1]:])

# Quick test — just to see raw output before we add any structure
print(generate("Extract the name and city from: John lives in Paris. Answer:"))

**Notice:** the raw output above is probably messy — extra words, no real JSON, maybe cut off mid-sentence. That messiness is normal and expected for GPT-2 with no guardrails. That's what the rest of this section fixes.

In [ ]:
# Step A: try to find and parse a JSON object inside a block of text.
# Why "find inside" instead of "parse the whole string"? Because the model
# often adds extra words before/after the JSON — we only want the JSON part.
def extract_json(text):
    """Look for {...} inside `text` and try to parse it. Returns a dict, or None if it fails."""
    match = re.search(r"\{.*\}", text, re.DOTALL)  # find the first { ... } block
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None  # looked like JSON but wasn't valid

# Try it on some example strings
print(extract_json('Sure! Here you go: {"name": "John", "city": "Paris"} Hope that helps!'))
print(extract_json('The name is John and he lives in Paris'))   # no JSON here → None

In [ ]:
# Step B: check that the parsed dict matches a simple schema.
# A "schema" here is just: which keys are required, and what type each should be.
def validate_schema(data, schema):
    """schema example: {"name": str, "city": str}
    Returns (True, "") if valid, or (False, "reason") if not."""
    if data is None:
        return False, "not valid JSON"
    for key, expected_type in schema.items():
        if key not in data:
            return False, f"missing key: {key}"
        if not isinstance(data[key], expected_type):
            return False, f"key '{key}' has wrong type"
    return True, "valid"

# Try it
schema = {"name": str, "city": str}
print(validate_schema({"name": "John", "city": "Paris"}, schema))   # should be valid
print(validate_schema({"name": "John"}, schema))                     # missing 'city'
print(validate_schema(None, schema))                                 # not JSON at all

In [ ]:
# Step C: put it all together into one pipeline function, with retries.
# "Retry" means: if the first attempt fails, try again (up to max_retries times)
# before giving up. This alone often rescues a surprising number of failures.
def structured_generate(prompt, schema, max_retries=3, **gen_kwargs):
    for attempt in range(max_retries):
        raw_output = generate(prompt, **gen_kwargs)
        data = extract_json(raw_output)
        is_valid, reason = validate_schema(data, schema)
        if is_valid:
            return {"success": True, "data": data, "attempts": attempt + 1, "raw": raw_output}
    return {"success": False, "data": None, "attempts": max_retries, "raw": raw_output}

# Test the full pipeline
prompt = 'Extract the name and city from this sentence as JSON: "John lives in Paris."\nJSON:'
result = structured_generate(prompt, schema={"name": str, "city": str})
print(result)

> **Checkpoint 1:** You now have a working pipeline: generate → extract → validate → retry. It might succeed or fail depending on luck this run — that inconsistency is *exactly* the problem the rest of this lab attacks.

**✏️ Exercise 1.1 (guided)** — Run the pipeline 5 times on the same prompt and count how many succeeded. This gives you a baseline "compliance rate" before any tuning. The code structure is given — just run it and read the result.

In [ ]:
# ✏️ Exercise 1.1 — this code is complete, just run it
n_runs = 5
successes = 0
for i in range(n_runs):
    result = structured_generate(prompt, schema={"name": str, "city": str})
    successes += result["success"]
    print(f"Run {i+1}: {'✅ success' if result['success'] else '❌ failed'}")

print(f"\nBaseline compliance rate: {successes}/{n_runs} = {successes/n_runs:.0%}")

---
## Part 2 — A/B Test Prompt Styles (15 min)

Slide 12 claimed four techniques improve compliance: being literal about format, delimiting input clearly, repeating constraints, and specifying negative space (what NOT to do). Let's test that claim instead of just believing it.

In [ ]:
# Four prompt variants for the SAME task — extracting name and city.
# Read each one and predict which will work best BEFORE running the test below.

task_sentence = "Maria works in Berlin."

prompt_variants = {
    "A: zero-shot, vague": (
        f'Extract the name and city from: {task_sentence}'
    ),
    "B: zero-shot, strict format": (
        f'Extract the name and city from this sentence and respond with ONLY this exact JSON format, '
        f'nothing else: {{"name": "...", "city": "..."}}\n'
        f'Sentence: {task_sentence}\nJSON:'
    ),
    "C: few-shot examples": (
        'Extract name and city as JSON.\n'
        'Sentence: "John lives in Paris." JSON: {"name": "John", "city": "Paris"}\n'
        'Sentence: "Amara works in Nairobi." JSON: {"name": "Amara", "city": "Nairobi"}\n'
        f'Sentence: "{task_sentence}" JSON:'
    ),
    "D: few-shot + negative space": (
        'Extract name and city as JSON. Output ONLY the JSON object — no explanation, no extra words.\n'
        'Sentence: "John lives in Paris." JSON: {"name": "John", "city": "Paris"}\n'
        'Sentence: "Amara works in Nairobi." JSON: {"name": "Amara", "city": "Nairobi"}\n'
        f'Sentence: "{task_sentence}" JSON:'
    ),
}

# ✏️ Your prediction before running anything:
# Which variant (A/B/C/D) do you predict will have the HIGHEST compliance rate? Why?
# your answer here as a comment: ...

In [ ]:
# Now let's actually measure it. This code is complete — just run it.
# We test each variant 6 times (schema-only compliance, temperature fixed at 0.7).
N_RUNS = 6
ab_results = {}

for name, prompt_text in prompt_variants.items():
    successes = 0
    for i in range(N_RUNS):
        result = structured_generate(prompt_text, schema={"name": str, "city": str}, temperature=0.7)
        successes += result["success"]
    ab_results[name] = successes / N_RUNS
    print(f"{name:<30} compliance: {successes}/{N_RUNS} = {successes/N_RUNS:.0%}")

In [ ]:
# 📊 Chart the comparison
plt.figure(figsize=(8, 4.5))
names = list(ab_results.keys())
values = [ab_results[n] * 100 for n in names]
plt.barh(names, values, color="#6C63FF")
plt.xlabel("Compliance rate (%)")
plt.title("Prompt style vs JSON compliance rate")
plt.xlim(0, 100)
plt.tight_layout(); plt.show()

> **Checkpoint 2:** Was your prediction right? Few-shot variants (C, D) usually beat zero-shot ones by a wide margin — this matches Slide 11's claim that examples are a stronger format signal than instructions.

**✏️ Exercise 2.1 (guided)** — Write ONE more prompt variant of your own (call it "E"), combining any techniques you like, and test it the same way. A starter template is below — just fill in the prompt text.

In [ ]:
# ✏️ Exercise 2.1 — fill in your own prompt text below, then run this cell
your_prompt = (
    "..."   # <-- replace this with your own prompt design
    f'{task_sentence}'
)

successes = 0
for i in range(N_RUNS):
    result = structured_generate(your_prompt, schema={"name": str, "city": str}, temperature=0.7)
    successes += result["success"]
print(f"Your variant E compliance: {successes}/{N_RUNS} = {successes/N_RUNS:.0%}")

---
## Part 3 — Tune Parameters (15 min)

Now fix the BEST prompt from Part 2, and vary temperature instead. Slide 14 claimed compliance should improve as temperature drops toward 0. Let's check.

In [ ]:
# Use your best-performing prompt variant from Part 2 here.
# (If unsure, variant "D" is a safe, strong default.)
best_prompt = prompt_variants["D: few-shot + negative space"]

temperatures = [0.1, 0.3, 0.5, 0.7, 1.0, 1.3]
temp_results = []

for temp in temperatures:
    successes = 0
    for i in range(N_RUNS):
        result = structured_generate(best_prompt, schema={"name": str, "city": str}, temperature=temp)
        successes += result["success"]
    rate = successes / N_RUNS
    temp_results.append(rate)
    print(f"T={temp:<4} compliance: {successes}/{N_RUNS} = {rate:.0%}")

In [ ]:
# 📊 The chart — this is what Slide 19's placeholder is asking for
plt.figure(figsize=(7, 4.5))
plt.plot(temperatures, [r*100 for r in temp_results], "o-", color="#FF6B4A", linewidth=2)
plt.xlabel("Temperature"); plt.ylabel("Compliance rate (%)")
plt.title("JSON compliance rate vs temperature")
plt.ylim(-5, 105); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

> **Checkpoint 3:** You should see a general downward trend as temperature rises (higher temperature = less compliance) — though with only 6 runs per point, expect some noise. This is the trade-off named on Slide 14: lower temperature trades away creativity for reliability.

**✏️ Exercise 3.1 (guided)** — Try the same sweep but with `top_p=0.5` instead of the default `0.9` (tighter nucleus, per Slide 14). Does tightening top_p help compliance even at higher temperatures? The code is mostly given — just add `top_p=0.5` to the `structured_generate(...)` call below.

In [ ]:
# ✏️ Exercise 3.1 — add top_p=0.5 to the function call below, then run it
tight_topp_results = []
for temp in temperatures:
    successes = 0
    for i in range(N_RUNS):
        result = structured_generate(best_prompt, schema={"name": str, "city": str}, temperature=temp)  # TODO: add top_p=0.5 here
        successes += result["success"]
    tight_topp_results.append(successes / N_RUNS)
    print(f"T={temp:<4} (tight top_p) compliance: {successes/N_RUNS:.0%}")

---
## Part 4 — Reliability Report (5 min)

Combine your findings into a short report. This is good practice for your assignment, which asks for something similar but for a task you design yourself.

**Fill in the blanks:**

- Best-performing prompt variant: ______ (compliance rate: ____%)
- Best-performing temperature: ______ (compliance rate: ____%)
- Combined (best prompt + best temperature), final compliance rate: ____%
- One thing that surprised you: ______

*(Double-click this cell to edit it directly.)*

---
## 🏆 Challenges (optional, ranked)

1. **Logit bias in practice (⭐⭐)** — Use `model.generate(..., bad_words_ids=...)` to ban a specific word (e.g., ban the model from ever generating "unknown") and confirm it never appears in 10 runs.
2. **A stricter schema (⭐⭐)** — Extend the schema to 3 fields (add `"age": int`) and re-run the Part 2 A/B test. Does compliance drop with more required fields? By how much?
3. **Stop sequences (⭐⭐⭐)** — Add a stop sequence so generation halts right after the closing `}` of the JSON object (hint: `model.generate` accepts `eos_token_id` tricks, or post-process by truncating at the first balanced `}`). Does this reduce trailing junk text?

In [ ]:
# 🏆 Challenge workspace
# your code here

---
## Summary

| Concept | What you did |
|---|---|
| Pipeline | Built generate → extract → validate → retry from scratch, understanding each step |
| A/B testing | Measured (not assumed) that few-shot + negative-space prompts dramatically improve compliance |
| Parameter tuning | Charted compliance vs temperature, confirming the reliability/creativity trade-off |
| Reliability reporting | Practiced summarizing evidence into a short, actionable report |

**Next:** the assignment ("The Reliability Report") asks you to do this full process for a task you design yourself.

*Module 3 · Session 3 — Generative & Agentic AI Systems*